# Lab Tracking trên Colab (GPU)

Chọn **Runtime → Change runtime type → T4 GPU**, rồi **Runtime → Run all**. Cả notebook chạy khoảng 40–60 phút.

Notebook làm thay các lệnh trong `HUONG_DAN.md`: cài đặt, tải dữ liệu, thử tracker, chạy bản nộp đủ frame, chấm `video_1`.

- `video_1` có nhãn: notebook **tự chọn** cấu hình có HOTA cao nhất.
- `video_2` … `video_5` không có nhãn: notebook chạy bản nộp với cấu hình mặc định ở mục 7. Sau khi xem video ở mục 6, sửa `CONFIG_MAT` rồi chạy lại từ mục 7.

## 1. Cài đặt

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout or "KHÔNG có GPU - đổi runtime sang T4")

In [ ]:
import os
import subprocess

# CHỈNH nếu repo hoặc nhánh của bạn khác.
REPO_BRANCH = "colab-lab"
REPO_URL = "https://github.com/Doan0904/K4-Track4-2A202602927-DangDinhDoan-Day22-Tracking-Student.git"

# Colab dùng Python 3.13, còn boxmot==10.0.42 ghim numpy==1.23.1. Bản này không có cho 3.13,
# và opencv-python cho Python >= 3.11 lại đòi numpy >= 1.23.5. Python 3.10 thì cả hai khớp,
# nên tạo môi trường 3.10 riêng bằng uv. Ô này chạy lại được nhiều lần.
%cd /content
!rm -rf /content/venv /content/lab /content/TrackEval
!pip install -q uv
!uv venv /content/venv --python 3.10 --clear
!git clone -q -b {REPO_BRANCH} {REPO_URL} /content/lab
!git clone -q https://github.com/JonathonLuiten/TrackEval.git /content/TrackEval
!uv pip install --python /content/venv/bin/python -r /content/lab/requirements.txt
!uv pip install -q --python /content/venv/bin/python -e /content/TrackEval
# boxmot 10.0.42 còn import pkg_resources, đã bị gỡ khỏi setuptools từ bản 81.
!uv pip install -q --python /content/venv/bin/python "setuptools<81"

# Mọi lệnh "python ..." phía sau (kể cả trong subprocess) sẽ dùng môi trường này.
os.environ["PATH"] = "/content/venv/bin:" + os.environ["PATH"]
# Ẩn cảnh báo pkg_resources / np.float vô hại để log dễ đọc.
os.environ["PYTHONWARNINGS"] = "ignore::UserWarning,ignore::DeprecationWarning"
# Colab đặt MPLBACKEND cho kernel, venv không có backend đó nên TrackEval lỗi khi vẽ biểu đồ.
os.environ["MPLBACKEND"] = "Agg"
%cd /content/lab
!python --version

# Dừng ngay nếu cài chưa đủ, thay vì để lỗi lộ ra ở các ô sau.
for module in ["cv2", "pytest", "trackeval", "boxmot", "ultralytics", "torch"]:
    result = subprocess.run(["python", "-c", f"import {module}"], capture_output=True, text=True)
    print(f"{module:12} {'OK' if result.returncode == 0 else 'LỖI'}")
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError(f"Chưa import được {module}")
print("Cài đặt OK")

Ô trên phải in `Python 3.10.x` rồi `Cài đặt OK`. Lần cài đầu tải torch nên mất vài phút. Nếu lỗi, dán toàn bộ phần output có chữ `error` hoặc `No solution` cho trợ lý. Ô này xóa và cài lại từ đầu mỗi lần chạy.


## 2. Dữ liệu

In [ ]:
%cd /content/lab
# CHỈNH: nếu bạn đã tự tải data_lab21.zip và để trên Google Drive, điền đường dẫn, ví dụ
# "/content/drive/MyDrive/data_lab21.zip". Để trống thì tải bằng gdown từ link của giảng viên.
DATA_ZIP_ON_DRIVE = ""

!mkdir -p /content/unzipped
if DATA_ZIP_ON_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    !unzip -q -o "{DATA_ZIP_ON_DRIVE}" -d /content/unzipped
else:
    # Truyền thẳng file ID: gdown bản mới (cài kèm boxmot trong venv) đã bỏ cờ --fuzzy.
    !gdown 1UeVPQd6j5pSzxoJDcKJrerT9SL3vJLDt -O /content/data_lab21.zip
    !unzip -q -o /content/data_lab21.zip -d /content/unzipped

In [ ]:
import os
from pathlib import Path

# Tìm thư mục chứa video_1 ... video_5. Bỏ qua __MACOSX (zip làm trên Mac có thư mục này,
# bên trong chỉ có file rác) và chỉ nhận thư mục thật sự có img1.
hits = sorted(p for p in Path("/content/unzipped").rglob("video_1")
              if "__MACOSX" not in p.parts and (p / "img1").is_dir())
assert hits, "Không thấy video_1/img1 trong dữ liệu đã giải nén"
LAB_DATA = str(hits[0].parent)
os.environ["LAB_DATA"] = LAB_DATA

# Gói dữ liệu hiện tại không kèm video_1/eval_config.json mà evaluate_practice.py cần.
# Tên benchmark chỉ dùng để đặt tên thư mục trong TrackEval, không đổi cách chấm.
eval_cfg = Path(LAB_DATA) / "video_1" / "eval_config.json"
if not eval_cfg.exists():
    eval_cfg.write_text('{"benchmark": "LAB", "split": "train"}\n')
    print("Đã tạo", eval_cfg)
print("LAB_DATA =", LAB_DATA)
!python scripts/check_data.py --lab-data-root "$LAB_DATA"
!python -m pytest -q

## 3. Hàm hỗ trợ

`show_previews` thu nhỏ video xem thử sang H.264 rồi nhúng vào notebook, kèm tên cấu hình.

In [ ]:
import base64
import subprocess
from IPython.display import HTML, display

DEVICE = "cuda:0"

def show_video(mp4, title, width=480):
    """Thu nhỏ video sang H.264 rồi nhúng vào notebook (Colab không phát được đường dẫn tương đối)."""
    mp4 = Path(mp4)
    web = mp4.with_name(mp4.stem + "_web.mp4")
    if not web.exists():
        subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", str(mp4),
                        "-vf", "scale=640:-2", "-vcodec", "libx264", "-crf", "32",
                        "-pix_fmt", "yuv420p", str(web)], check=True)
    data = base64.b64encode(web.read_bytes()).decode()
    display(HTML(f"<b>{title}</b><br><video width={width} controls loop "
                 f"src='data:video/mp4;base64,{data}'></video>"))

def show_previews(root, video):
    for mp4 in sorted(Path(root).glob(f"{video}_*/{video}_preview.mp4")):
        show_video(mp4, mp4.parent.name)

## 4. Baseline (bước D)

`video_1` + ByteTrack, 150 frame đầu.

In [ ]:
!python scripts/run_tracking.py --source "$LAB_DATA/video_1/img1" --seq-name video_1 \
  --tracker bytetrack --conf 0.3 --iou 0.5 --out runs/thu_nhanh --save-video --max-frames 150 --device {DEVICE}

## 5. `video_1`: chấm số (có nhãn)

**5a.** Cả 5 tracker ở `conf` 0.3, `iou` 0.5, chạy đủ frame, có HOTA / MOTA / IDF1.

In [ ]:
!python scripts/sweep_tracking.py --lab-data-root "$LAB_DATA" --video video_1 \
  --trackers bytetrack ocsort botsort strongsort deepocsort \
  --device {DEVICE} --trackeval-root /content/TrackEval --no-video \
  --out runs/sweep --save-json runs/sweep/video_1_5a.json

**5b.** Lấy tracker có HOTA cao nhất ở 5a, quét `conf` 0.15 / 0.3 / 0.5 và `iou` 0.4 / 0.5 / 0.7, mỗi lần đổi một số.

In [ ]:
import json

def best_by_hota(path):
    rows = [r for r in json.loads(Path(path).read_text()) if "HOTA" in r]
    assert rows, f"Không có số HOTA trong {path} - xem log chấm phía trên"
    return max(rows, key=lambda r: r["HOTA"])

BEST_V1 = best_by_hota("runs/sweep/video_1_5a.json")["tracker"]
print("Tracker tốt nhất ở 5a:", BEST_V1)

!python scripts/sweep_tracking.py --lab-data-root "$LAB_DATA" --video video_1 \
  --trackers {BEST_V1} --sweep-trackers {BEST_V1} \
  --device {DEVICE} --trackeval-root /content/TrackEval --no-video \
  --out runs/sweep --save-json runs/sweep/video_1_5b.json

best = best_by_hota("runs/sweep/video_1_5b.json")
V1_CONFIG = (best["tracker"], best["conf"], best["iou"])
print("Cấu hình video_1 chọn để nộp:", V1_CONFIG, f"HOTA={best['HOTA']:.2f}")

## 6. `video_2` … `video_5`: xem bằng mắt

**6a.** Mỗi video chạy cả 5 tracker ở `conf` 0.3, `iou` 0.5, 150 frame đầu, có video. Khi xem, ghi lại:

- ID có đổi màu giữa chừng không (đổi danh tính);
- hộp có nhảy sang người bên cạnh khi hai người cắt nhau không;
- có người rõ mà không có hộp không (`conf` cao quá);
- có hộp nhấp nháy trên nền, bóng, kính không (`conf` thấp quá).

In [ ]:
EYE_VIDEOS = ["video_2", "video_3", "video_4", "video_5"]

for v in EYE_VIDEOS:
    !python scripts/sweep_tracking.py --lab-data-root "$LAB_DATA" --video {v} \
      --trackers bytetrack ocsort botsort strongsort deepocsort \
      --device {DEVICE} --max-frames 150 --out runs/eye

In [ ]:
for v in EYE_VIDEOS:
    display(HTML(f"<h3>{v}</h3>"))
    show_previews("runs/eye", v)

**6b.** Với tracker bạn thấy tốt nhất ở mỗi video, quét `conf` / `iou` (vẫn 150 frame). Sửa `PICKS` sau khi xem 6a, rồi chạy lại ô này.

In [ ]:
PICKS = {  # CHỈNH sau khi xem 6a
    "video_2": "botsort",
    "video_3": "botsort",
    "video_4": "botsort",
    "video_5": "botsort",
}

for v, pick in PICKS.items():
    !python scripts/sweep_tracking.py --lab-data-root "$LAB_DATA" --video {v} \
      --trackers {pick} --sweep-trackers {pick} \
      --device {DEVICE} --max-frames 150 --out runs/eye_sweep

In [ ]:
for v in PICKS:
    display(HTML(f"<h3>{v}</h3>"))
    show_previews("runs/eye_sweep", v)

**6c.** Bảng số không cần nhãn cho các lần chạy 150 frame ở 6a và 6b: số hộp mỗi frame, số ID, độ dài trung vị của track, tỉ lệ track ngắn (< 15 frame) và tỉ lệ track bị đứt quãng. Đây chỉ là chỉ báo phụ: ít ID có thể do tracker tốt, cũng có thể do detector bỏ sót người. Dùng bảng để biết cặp video nào cần xem kỹ, không dùng thay cho việc xem.

In [ ]:
!git pull -q
for v in EYE_VIDEOS:
    print(f"\n##### {v}")
    !python scripts/track_stats.py --runs-dir runs/eye runs/eye_sweep | grep -E "lần chạy|{v}_"

## 7. Bản nộp: chạy đủ frame

`video_1` lấy cấu hình tự chọn ở 5b. Bốn video kia lấy `CONFIG_MAT`: giá trị dưới đây chỉ là điểm xuất phát, **chưa phải kết quả đã kiểm chứng**. Sửa theo điều bạn thấy ở mục 6, rồi chạy lại từ ô này xuống.

In [ ]:
CONFIG_MAT = {  # CHỈNH: (tracker, conf, iou)
    "video_2": ("botsort", 0.3, 0.5),
    "video_3": ("botsort", 0.3, 0.5),
    "video_4": ("botsort", 0.3, 0.5),
    "video_5": ("botsort", 0.3, 0.5),
}
CONFIG = {"video_1": V1_CONFIG, **CONFIG_MAT}

for video, (tracker, conf, iou) in CONFIG.items():
    print(f"\n##### {video}: {tracker} conf={conf} iou={iou}")
    !python scripts/run_tracking.py --source "$LAB_DATA/{video}/img1" --seq-name {video} \
      --tracker {tracker} --conf {conf} --iou {iou} --out runs/nop_bai --save-video --device {DEVICE}

Kiểm tra đủ frame: frame cuối có hộp nên bằng (hoặc rất gần) số ảnh trong `img1`.

In [ ]:
for video in CONFIG:
    n_imgs = len(list((Path(LAB_DATA) / video / "img1").glob("*.jpg")))
    out = Path(f"runs/nop_bai/{video}.txt")
    assert out.exists(), f"Thiếu {out} - xem log của {video} ở ô trên"
    rows = out.read_text().splitlines()
    last = max(int(r.split(",")[0]) for r in rows) if rows else 0
    print(f"{video}: {n_imgs} ảnh, {len(rows)} dòng, frame cuối có hộp = {last}")
Path("runs/nop_bai/cau_hinh.json").write_text(json.dumps(CONFIG, indent=2))

## 8. Chấm `video_1` cho báo cáo

In [ ]:
!python scripts/evaluate_practice.py --trackeval-root /content/TrackEval \
  --lab-data-root "$LAB_DATA" --submission runs/nop_bai/video_1.txt --run-name nop_bai_video1

Xem lại video bản nộp của cả năm video để viết quan sát cho báo cáo.

In [ ]:
for video in CONFIG:
    show_video(f"runs/nop_bai/{video}_preview.mp4", video)

## 9. Tải kết quả về

`nop_bai_txt.zip` gồm 5 file `.txt`, `cau_hinh.json` và các bảng quét. Video xem thử đủ frame khá nặng nên để riêng trong `nop_bai_video.zip` (tải nếu cần).

In [ ]:
!cd runs && zip -q -j nop_bai_txt.zip nop_bai/*.txt nop_bai/cau_hinh.json sweep/*.json
!cd runs && zip -q -j nop_bai_video.zip nop_bai/*_preview.mp4 -x "*_web.mp4"
from google.colab import files
files.download("runs/nop_bai_txt.zip")
# files.download("runs/nop_bai_video.zip")  # bỏ dấu # nếu muốn tải video